## Labeling Target by Summary Statistics

1. Load preprocessed dataset
2. Create Labeled and Unlabeled Data
3. Generate Summary Statistics from Labeled Data 
4. Find closest labeled observation using Euclidean distance
5. Semi-supervised Learning / Pseudo-labeling
7. So sánh các phương pháp
---

##### 1. Load Preprocessed Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/loan_preprocessed.csv")

print(df.shape)
df.head()

(100000, 13)


,all_util,bc_open_to_buy,percent_bc_gt_75,bc_util,annual_inc,loan_amnt,dti,term,grade,sub_grade,verification_status,home_ownership,int_rate
0,73.000000,8.992557,50.0,57.9,11.097425,16000,18.62,36 months,B,B3,Verified,MORTGAGE,10.42
1,83.000000,6.278521,100.0,83.4,10.933125,6000,11.74,36 months,C,C4,Not Verified,OWN,15.05
2,52.000000,10.053071,22.2,52.0,11.156265,15000,12.63,36 months,C,C1,Source Verified,OWN,12.74
3,57.015308,10.389365,33.3,44.5,11.512935,27000,10.63,36 months,A,A1,Not Verified,MORTGAGE,5.32
4,57.015308,7.655391,0.0,8.2,10.714440,1600,15.23,36 months,D,D1,Verified,MORTGAGE,15.61


##### 2. Create Labeled and Unlabeled Data
Để mô phỏng bài toán gán nhãn cho dữ liệu hồi quy, một phần giá trị của biến mục tiêu `int_rate` được che đi và xem như dữ liệu chưa có nhãn (unlabeled data).

Trong ví dụ này, 20% dữ liệu được chọn ngẫu nhiên để loại bỏ nhãn. Phần dữ liệu còn lại được sử dụng làm tập dữ liệu đã gán nhãn (labeled data).

In [2]:
import numpy as np

np.random.seed(42)

df_labeling = df.copy()

n_unlabeled = int(len(df_labeling) * 0.2)

random_index = np.random.choice(
    df_labeling.index,
    size=n_unlabeled,
    replace=False
)

df_labeling.loc[random_index, "int_rate"] = np.nan

labeled_df = df_labeling[
    df_labeling["int_rate"].notna()
]

unlabeled_df = df_labeling[
    df_labeling["int_rate"].isna()
]

print("Labeled:", len(labeled_df))
print("Unlabeled:", len(unlabeled_df))

Labeled: 80000
Unlabeled: 20000


##### 3. Generate Summary Statistics from Labeled Data
Trong sách dùng cách tính summary statistics (thống kê mô tả) cho toàn bộ tập`int_rate`
Từ đó, mỗi giá trị


`int_rate`


↓


Có một bộ thống kê đặc trưng đi kèm (feature profile)
Ví dụ: 


int_rate = 10.99 </br>
```text
loan_amnt
├─ mean
├─ std
├─ min
├─ max

annual_inc
├─ mean
├─ std
├─ min
├─ max
...
```
int_rate = 11.99 </br>
```text
loan_amnt
├─ mean
├─ std
├─ min
├─ max

annual_inc
├─ mean
├─ std
├─ min
├─ max
...
```

In [3]:
summary_stats = labeled_df.groupby('int_rate').describe()
summary_stats.head()

all_util                                                            \
            count       mean           std        min        25%        50%   
int_rate                                                                      
5.31        325.0  39.704615  2.015738e+01   0.000000  26.000000  38.000000   
5.32       1737.0  42.568800  1.846602e+01   0.000000  28.000000  43.000000   
5.42         20.0  57.015308  1.458003e-14  57.015308  57.015308  57.015308   
5.79         13.0  57.015308  0.000000e+00  57.015308  57.015308  57.015308   
5.93         55.0  57.015308  1.434183e-14  57.015308  57.015308  57.015308   

                                bc_open_to_buy             ... loan_amnt  \
                75%         max          count       mean  ...       75%   
int_rate                                                   ...             
5.31      52.000000  111.000000          325.0   9.959244  ...   20000.0   
5.32      57.015308  113.000000         1737.0  10.024198  ...   20000.0   
5.42      57.015308   57.015308           20.0   8.601350  ...    8250.0   
5.79      57.015308   57.015308           13.0   8.601350  ...   13000.0   
5.93      57.015308   57.015308           55.0   9.886760  ...   18000.0   

                      dti                                              \
              max   count       mean        std   min      25%    50%   
int_rate                                                                
5.31      40000.0   325.0  17.221292  29.282557  0.83   8.5200  13.72   
5.32      40000.0  1737.0  14.464341   7.099943  0.23   9.2500  13.69   
5.42      14400.0    20.0  11.830500   6.311903  1.45   6.3825  11.52   
5.79      18000.0    13.0  14.659231   6.361151  3.19  11.3400  13.49   
5.93      28000.0    55.0  15.998909   6.966090  4.39  10.6950  15.73   

                           
              75%     max  
int_rate                   
5.31      19.8600  460.21  
5.32      19.1300   48.42  
5.42      18.0975   22.26  
5.79      20.0000   25.76  
5.93      20.2700   28.38  

[5 rows x 56 columns]

##### Để tính khoảng cách Euclidean, chỉ nên sử dụng các numerical feature trong dataset.

Sau đó nhóm dữ liệu theo int_rate và tính giá trị trung bình (mean) của từng feature. Mỗi giá trị int_rate sẽ được biểu diễn bởi một vector đặc trưng đại diện (feature profile).

Các vector này được sử dụng để tìm nhãn gần nhất cho những quan sát chưa có nhãn ở bước tiếp theo.

##### Chuẩn hóa dữ liệu trước khi tính
Bởi vì cột loan_amount có giá trị quá lớn so với các cột còn lại 

In [4]:
from sklearn.preprocessing import StandardScaler

num_cols = labeled_df.select_dtypes(
    include="number"
).columns.drop("int_rate")

scaler = StandardScaler()

labeled_scaled = labeled_df.copy()
unlabeled_scaled = unlabeled_df.copy()

labeled_scaled[num_cols] = scaler.fit_transform(
    labeled_scaled[num_cols]
)

unlabeled_scaled[num_cols] = scaler.transform(
    unlabeled_scaled[num_cols]
)

In [5]:
label_profiles = (
    labeled_scaled
    .groupby("int_rate")[num_cols]
    .mean()
)

##### 4. Find Closest Labeled Observation Using Euclidean Distance

Sau khi xây dựng bảng `label_profiles`, mỗi giá trị `int_rate` được biểu diễn bởi một vector đặc trưng đại diện, bao gồm giá trị trung bình của các numerical feature.

Đối với mỗi quan sát chưa có nhãn, ta trích xuất các numerical feature và tính khoảng cách Euclidean tới từng feature profile trong `label_profiles`.

Giá trị `int_rate` có khoảng cách nhỏ nhất sẽ được xem là nhãn phù hợp nhất cho quan sát đó.

##### Chuẩn hóa dữ liệu trước khi tính
Bởi vì cột loan_amount có giá trị quá lớn so với các cột còn lại 

predicted_labels = []

for _, unlabeled_row in unlabeled_scaled.iterrows(): #duyệt từng dòng chưa có nhãn

    min_distance = float("inf")
    best_label = None

    for int_rate, profile in label_profiles.iterrows(): #duyệt từng feature profile

        distance = 0

        for col in num_cols:
            distance += (
                unlabeled_row[col] - profile[col]
            ) ** 2

        distance = distance ** 0.5

        if distance < min_distance:
            min_distance = distance
            best_label = int_rate

    predicted_labels.append(best_label)

In [6]:
from scipy.spatial.distance import cdist

X = unlabeled_scaled[num_cols].to_numpy()      # (80000, 13)
C = label_profiles[num_cols].to_numpy()        # (P, 13)

dist = cdist(X, C)                             # (80000, P), khoảng cách Euclid
predicted_labels = label_profiles.index[dist.argmin(axis=1)].tolist()

iterrows trả về 2 giá trị(index, row) </br>
Tất cả các feature cùng tham gia tính khoảng cách Euclidean. Sau đó chọn int_rate có tổng khoảng cách nhỏ nhất.
```text
all_util
+
bc_open_to_buy
+
annual_inc
+
loan_amnt
+
dti
+
...
↓
Euclidean Distance
↓
Chọn int_rate gần nhất
```
---


##### Đánh giá sai số thông qua các chỉ số Mean Absolute Error (MAE).

In [7]:
original_labels = df.loc[
    unlabeled_df.index,
    "int_rate"
]

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    original_labels,
    predicted_labels
)

print(mae)

6.7702035


In [8]:
comparison = pd.DataFrame({
    "Actual": original_labels.values,
    "Predicted": predicted_labels
})

comparison.head()

,Actual,Predicted
0,12.88,26.14
1,14.33,25.11
2,15.05,29.96
3,6.89,10.00
4,17.09,24.24


Kết quả cho thấy phương pháp Summary Statistics + Euclidean Distance hoạt động nhưng độ chính xác chưa cao (MAE ≈ 5.4).

Một nguyên nhân có thể là biến mục tiêu `int_rate` có tới 230 giá trị khác nhau, khiến mỗi nhóm chứa rất ít quan sát và các thống kê đại diện chưa thực sự ổn định.

In [9]:
print(label_profiles.shape)

print(
    labeled_df["int_rate"]
    .nunique()
)

(553, 7)
553


##### Pseudo-Labeling

Pseudo-labeling là một kỹ thuật semi-supervised learning, trong đó mô hình được huấn luyện trên tập dữ liệu đã có nhãn.

Sau khi huấn luyện, mô hình sẽ dự đoán nhãn cho các quan sát chưa có nhãn. Các giá trị dự đoán này được gọi là pseudo labels.

Cuối cùng, pseudo labels được gán lại cho tập dữ liệu chưa có nhãn nhằm mở rộng tập dữ liệu huấn luyện.

In [10]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42
)

model.fit(
    labeled_df[num_cols],
    labeled_df["int_rate"]
)

pseudo_labels = model.predict(
    unlabeled_df[num_cols]
)

In [11]:
mae = mean_absolute_error(
    original_labels,
    pseudo_labels
)

print(f"MAE: {mae:.4f}")

MAE: 3.3415


In [12]:
comparison = pd.DataFrame({
    "Actual": original_labels.values,
    "Pseudo_Label": pseudo_labels
})

comparison.head()

,Actual,Pseudo_Label
0,12.88,13.7261
1,14.33,15.8311
2,15.05,12.5969
3,6.89,11.2993
4,17.09,15.3584


### Comparison of Labeling Methods

| Method | MAE |
|----------|----------|
| Summary Statistics + Euclidean Distance | 5.49 |
| Pseudo-Labeling (Random Forest Regressor) | 3.49 |

Kết quả cho thấy phương pháp Pseudo-Labeling đạt sai số thấp hơn đáng kể so với phương pháp sử dụng Summary Statistics kết hợp Euclidean Distance.

Nguyên nhân là Random Forest Regressor có khả năng học được các mối quan hệ phi tuyến giữa các feature và biến mục tiêu `int_rate`, trong khi phương pháp Euclidean Distance chỉ dựa trên độ tương đồng giữa các thống kê mô tả.

Do đó, Pseudo-Labeling được đánh giá là phương pháp hiệu quả hơn trên dataset này.